# Treino no Colab: QLoRA (Qwen3.5-2B), Laya e BERTimbau

1. No Mac: `make colab-bundle` e suba `dist/colab_bundle.zip` para o seu Google Drive (qualquer pasta).
2. Aqui: *Ambiente de execução → Alterar tipo → GPU* (A100 ou L4) e rode as células **em ordem**.
3. Os treinos gravam direto em **MyDrive/review-intelligence-ptbr/outputs/**, então se a sessão cair nada se perde. No fim, baixe as pastas de `outputs/` para o Mac (pasta `outputs/` do repo) ou rode `make import-models ZIP=...`.

Se o ambiente reiniciar, rode de novo a célula 1 (Drive + unzip) e continue de onde parou.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive e pacote

In [ ]:
import glob, os, subprocess
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/review-intelligence-ptbr'
os.makedirs(OUT, exist_ok=True)

# procura o pacote em qualquer lugar do Drive (o Drive às vezes renomeia para 'colab_bundle (1).zip')
found = sorted(glob.glob('/content/drive/MyDrive/**/colab_bundle*.zip', recursive=True), key=os.path.getmtime)
assert found, 'colab_bundle.zip não está no Drive: suba dist/colab_bundle.zip para o MyDrive'
BUNDLE = found[-1]
print('usando', BUNDLE)
subprocess.run(['rm', '-rf', '/content/ri'], check=True)
subprocess.run(['unzip', '-q', BUNDLE, '-d', '/content/ri'], check=True)
os.chdir('/content/ri')
assert os.path.exists('pyproject.toml') and os.path.exists('data/labels/silver.jsonl')
print(sorted(os.listdir('.')))

In [ ]:
%cd /content/ri
# torch já vem no Colab; o pacote e as dependências de treino entram por cima.
# flash-linear-attention: kernels Triton da atenção linear do Qwen3.5 (instala rápido, sem compilar).
# causal-conv1d fica de fora de propósito: compila CUDA por 10-30 min e o ganho é pequeno.
!pip install -q -e ".[colab,dev]" flash-linear-attention

## 2. Testes e smoke na GPU
Valida o caminho CUDA (QLoRA 4 bits) com 5 passos antes do treino completo.

In [ ]:
%cd /content/ri
!python -m pytest -q
!python -m review_intel.training.qlora --backend cuda --model Qwen/Qwen3.5-2B --labels data/labels/silver.jsonl --limit 64 --max-steps 5 --warmup 1 --output /content/smoke-qlora

## 3. QLoRA do Qwen3.5-2B (~9 mil exemplos, 2 épocas)

In [ ]:
%cd /content/ri
!python -m review_intel.training.qlora --backend cuda --model Qwen/Qwen3.5-2B --labels data/labels/silver.jsonl --epochs 2 --batch-size 8 --grad-accum 2 --output /content/drive/MyDrive/review-intelligence-ptbr/outputs/lora-Qwen3.5-2B

## 4. Laya multilingual

In [ ]:
%cd /content/ri
!python -m review_intel.training.laya --labels data/labels/silver.jsonl --device cuda --output /content/drive/MyDrive/review-intelligence-ptbr/outputs/laya-ft

## 5. BERTimbau multi-aspecto

In [ ]:
%cd /content/ri
!python -m review_intel.training.bertimbau --labels data/labels/silver.jsonl --output /content/drive/MyDrive/review-intelligence-ptbr/outputs/bertimbau-multiaspecto

## 6. Conferência e envio para o Drive
O zip leva só os artefatos finais (sem checkpoints nem os JSONL de treino do Laya).

In [ ]:
%cd /content/ri
# Os treinos já gravam direto em MyDrive/review-intelligence-ptbr/outputs/: se a sessão cair, nada se perde.
!ls -lh /content/drive/MyDrive/review-intelligence-ptbr/outputs/*/
# Para o Mac: baixe cada pasta do Drive. Pastas grandes vêm divididas em vários zips (-1-001, -1-002, ...):
# baixe TODAS as partes, ou baixe os arquivos de pesos (model.safetensors, model.pt, adapter_model.safetensors) um a um.